<a href="https://colab.research.google.com/github/laramalkawi81-ops/DS230-Instacart-Project/blob/main/Copy_of_04_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## DS230 Final Project _ File 4
## Task A: Reorder Prediction (Classification)

In this notebook, we train and evaluate multiple classification models
to predict whether a user will reorder a product in their next order.

In [5]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/instacart_data"
data = pd.read_csv(f"{DATA_PATH}/model_data.csv")


We define the classification target as `reordered`
and exclude idenfifiers to avoid leakage.

In [ ]:
target = 'reordered'

drop_cols = [ 'reordered', 'order_id', 'user_id', 'product_id']
X = data.drop(columns=drop_cols)
y = data[target]


A time-aware split is used to ensure that training data chronologically precedes validation data.

In [ ]:
split_point = data['order_number'].quantile(0.8)
X_train = X[data['order_number'] < split_point]
X_test = X[data['order_number'] >= split_point]

y_train = y[data['order_number'] < split_point]
y_train = y[data['order_number'] >= split_point]



Logistic Regression serves as a strong linear baseline with class weighting to handle imbalance.

In [ ]:
log_reg = LogisticRegression(
    max_iter=1000
    class_weight='balanced'

)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)
y_prob_lr = log_reg.predict_proba(X_test)[:, 1]

In [ ]:
print(classification_report(y_test, y_pred_lr))
print("RoC AUC:", roc_auc_score(y_test, y_prob_lr))


KNN captures local similarity patterns but is sensitive to feature scaling and computational cost.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=10)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)
y_prob_knn = knn.predict_proba(X_test)[:,1]

print("KNN ROC AUC", roc_auc_score(y_test, y_prob_knn))


SVM is tested with a linear kernel due to scalability constraints.

In [ ]:
svm = SVC(kernel='linear', probability=True, class_weight='balanced')
svm.fit(X_train, y_train)

y_prob_svm = svm.predict_proba(X_test)[:, 1]
print("SVM ROC AUC:", roc_auc_score(y_test, y_prob_svm))

Decision Trees capture non_linear interactions but may overfit without constraints.

In [ ]:
dt = DecisionTreeClassifier(
    max_depth=10,
    class_weight='balanced',
    random_state=42

)
dt.fit(X_train, y_train)
y_prob_dt = dt.predict_proba(X_test)[:,1]

print("Decision Tree ROC AUC:", roc_auc_score(y_test,y_prob_dt)))

Random Forest reduces variance  by aggregating multiple trees and serves as a strong ensemble baseline.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest ROC AUC:", roc_auc_score(y_test, y_prob_rf))

ROC curves are plotted to compare model discrimination ability

In [ ]:
models = {
    "Logistic Regression": y_prob_lr,
    "KNN": y_prob_knn,
    "SVM": y_prob_svm,
    "Decision Tree": y_prob_dt,
    "Random Forest": y_prob_rf
}

plt.figure(figsize=(7,5))

for name, probs in models.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=name)

plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

### Task A Summary

Multiple classification models were trained and evaluated using
time-aware splits. Ensemble methods, particularly Random Forest,
demonstrated superior performance in capturing complex reorder behavior.
These results motivate further tuning and explainability analysis
in subsequent stages.